In [ ]:
import json, shutil
from pathlib import Path
from datetime import datetime, timezone
import kagglehub          

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
KAGGLE_ID    = "moazawadali/plant-leaf-diseases"      
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Project root:", PROJECT_ROOT)
print("Raw dir     :", RAW_DIR)

In [ ]:
class DatasetDownloader:
    """Downloads a Kaggle dataset into data/raw and records its provenance."""

    def __init__(self, kaggle_id: str, raw_dir: Path):
        self.kaggle_id = kaggle_id
        self.raw_dir   = Path(raw_dir)
        self.target    = self.raw_dir / kaggle_id.split("/")[-1]   

    def download(self, force: bool = False) -> Path:
        # Skip the download if the folder already has files (saves time/bandwidth)
        if self.target.exists() and any(self.target.iterdir()) and not force:
            print(f"[skip] Already downloaded: {self.target}")
            return self.target

        self.raw_dir.mkdir(parents=True, exist_ok=True)

        # kagglehub downloads into its own cache and returns that path
        cache_path = Path(kagglehub.dataset_download(self.kaggle_id))
        print(f"[ok] Downloaded to cache: {cache_path}")

        # Copy into the project so data lives inside data/raw
        if self.target.exists():
            shutil.rmtree(self.target)
        shutil.copytree(cache_path, self.target)

        self._write_provenance(cache_path)
        return self.target

    def _write_provenance(self, cache_path: Path) -> None:
        # Keeps a record of where the data came from (useful for the paper's reference)
        info = {
            "kaggle_id":     self.kaggle_id,
            "url":           f"https://www.kaggle.com/datasets/{self.kaggle_id}",
            "downloaded_at": datetime.now(timezone.utc).isoformat(),
            "cache_path":    str(cache_path),
            "local_path":    str(self.target),
        }
        (self.raw_dir.parent / "provenance.json").write_text(json.dumps(info, indent=2))

In [ ]:
downloader   = DatasetDownloader(KAGGLE_ID, RAW_DIR)
DATASET_PATH = downloader.download()      # use force=True to re-download
print("Dataset path:", DATASET_PATH)

In [ ]:
class DatasetValidator:
    """Checks the path exists, finds the folder holding the class sub-folders,
    and reports image counts per class."""

    def __init__(self, root: Path, img_exts: set, min_classes: int = 2):
        self.root        = Path(root)
        self.img_exts    = img_exts
        self.min_classes = min_classes
        self.class_root  = None     # folder whose sub-folders are the classes
        self.counts      = {}       # {class_name: n_images}

    def _images_in(self, folder: Path) -> list:
        # Count images directly inside a folder (recursive, so nested class dirs still work)
        return [p for p in folder.rglob("*") if p.suffix.lower() in self.img_exts]

    def _find_class_root(self) -> Path:
        # The dataset may be nested (e.g. root/Plants/train/<classes>).
        # Pick the shallowest folder that has >= min_classes sub-folders containing images.
        candidates = [self.root] + sorted(
            (p for p in self.root.rglob("*") if p.is_dir()),
            key=lambda p: len(p.parts),
        )
        for folder in candidates:
            subs = [d for d in folder.iterdir() if d.is_dir() and self._images_in(d)]
            if len(subs) >= self.min_classes:
                return folder
        raise ValueError("No folder with class sub-folders containing images was found.")

    def validate(self) -> dict:
        # 1) path exists and is not empty
        assert self.root.exists(), f"Path not found: {self.root}"
        assert any(self.root.iterdir()), f"Path is empty: {self.root}"

        # 2) locate the class folders and count images per class
        self.class_root = self._find_class_root()
        for d in sorted(self.class_root.iterdir()):
            if d.is_dir():
                self.counts[d.name] = len(self._images_in(d))

        # 3) no empty class
        empty = [c for c, n in self.counts.items() if n == 0]
        assert not empty, f"Empty classes: {empty}"

        return self.counts

    def report(self) -> None:
        total = sum(self.counts.values())
        print(f"Class root : {self.class_root}")
        print(f"Classes    : {len(self.counts)}")
        print(f"Images     : {total}\n")
        for cls, n in self.counts.items():
            print(f"  {cls:<35} {n:>6}  ({n / total:6.1%})")
        # Rough imbalance indicator, relevant for the paper's "Limites" section
        ratio = max(self.counts.values()) / min(self.counts.values())
        print(f"\nImbalance ratio (max/min): {ratio:.2f}")

In [ ]:
validator = DatasetValidator(DATASET_PATH, IMG_EXTS)
validator.validate()
validator.report()
DATA_ROOT = validator.class_root